# SYNERGI - SYstemic NEuRal Graph Intelligence



A Hybrid Vision–Rule-Governed Intelligence Framework for Industrial Automation in Sugarcane Mills

## Modules

In [2]:
import pandas as pd
import numpy as np
import os

## Configs

In [6]:
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
ROOT_DIR

'e:\\Studies\\MIT\\8\\Project'

### Dataset Directory

In [14]:
DATA_DIR = os.path.join(ROOT_DIR, "Datasets", "severstal-steel-defect-detection")
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "train.csv")
(TRAIN_CSV_PATH, os.path.exists(TRAIN_CSV_PATH))


('e:\\Studies\\MIT\\8\\Project\\Datasets\\severstal-steel-defect-detection\\train.csv',
 True)

## Data Preprocessing

In [15]:
df = pd.read_csv(TRAIN_CSV_PATH)

In [17]:
df.head(10)

,ImageId,ClassId,EncodedPixels
0,0002cc93b.jpg,1,29102 12 29346 24 29602 24 29858 24 30114 24 3...
1,0007a71bf.jpg,3,18661 28 18863 82 19091 110 19347 110 19603 11...
2,000a4bcdd.jpg,1,37607 3 37858 8 38108 14 38359 20 38610 25 388...
3,000f6bf48.jpg,4,131973 1 132228 4 132483 6 132738 8 132993 11 ...
4,0014fce06.jpg,3,229501 11 229741 33 229981 55 230221 77 230468...
5,0025bde0c.jpg,3,8458 14 8707 35 8963 48 9219 71 9475 88 9731 8...
6,0025bde0c.jpg,4,315139 8 315395 15 315651 16 315906 17 316162 ...
7,002af848d.jpg,4,290800 6 291055 13 291311 15 291566 18 291822 ...
8,002fc4e19.jpg,1,146021 3 146275 10 146529 40 146783 46 147038 ...
9,002fc4e19.jpg,2,145658 7 145901 20 146144 33 146386 47 146629 ...


In [18]:
df.shape

(7095, 3)

In [20]:
df.dtypes

ImageId          object
ClassId           int64
EncodedPixels    object
dtype: object

### Image and Defect Class Relationship

Expected to have 4 entries for each image corresponding to each class of defect (1, 2, 3, 4), regardless of whether or not the particular image has a defect of said class ID.

In [23]:
df['ImageId'].value_counts().head(10)

ImageId
db4867ee8.jpg    3
ef24da2ba.jpg    3
ff6bfada2.jpg    2
012f26693.jpg    2
fe2234ba6.jpg    2
fdb7c0397.jpg    2
fd26ab9ad.jpg    2
fc4f997c5.jpg    2
fc2d2d29e.jpg    2
268f0fa63.jpg    2
Name: count, dtype: int64

In [37]:
print(f"Count of Images with not all categories of class defect data available: {int((df['ImageId'].value_counts() != 4).sum())}/{len(df)}")

Count of Images with not all categories of class defect data available: 6666/7095


### Creating Full Dataset

In [26]:
all_images = df['ImageId'].unique()
all_classes = [1, 2, 3, 4]

In [27]:
full_index = pd.MultiIndex.from_product(
    [all_images, all_classes],
    names=['ImageId', 'ClassId']
)

In [29]:
df_full = (
    df.set_index(['ImageId', 'ClassId'])
      .reindex(full_index)
      .reset_index()
)

In [41]:
df_full.shape

(26664, 3)

In [31]:
df_full.head(10)

,ImageId,ClassId,EncodedPixels
0,0002cc93b.jpg,1,29102 12 29346 24 29602 24 29858 24 30114 24 3...
1,0002cc93b.jpg,2,NaN
2,0002cc93b.jpg,3,NaN
3,0002cc93b.jpg,4,NaN
4,0007a71bf.jpg,1,NaN
5,0007a71bf.jpg,2,NaN
6,0007a71bf.jpg,3,18661 28 18863 82 19091 110 19347 110 19603 11...
7,0007a71bf.jpg,4,NaN
8,000a4bcdd.jpg,1,37607 3 37858 8 38108 14 38359 20 38610 25 388...
9,000a4bcdd.jpg,2,NaN


In [30]:
df_full['ImageId'].value_counts().head()

ImageId
0002cc93b.jpg    4
0007a71bf.jpg    4
000a4bcdd.jpg    4
000f6bf48.jpg    4
0014fce06.jpg    4
Name: count, dtype: int64

In [39]:
print(f"Count of images with not all categories of class defect data available: {int((df_full['ImageId'].value_counts() != 4).sum())}/{len(df)}")

Count of images with not all categories of class defect data available: 0/7095


In [42]:
df_full['EncodedPixels'].isna().mean() * 100

np.float64(73.39108910891089)

### Essentials

In [57]:
image_level_defects = (
    df_full.groupby('ImageId')['EncodedPixels']
           .apply(lambda x: x.notna().any())
)
image_level_defects.value_counts()

EncodedPixels
True    6666
Name: count, dtype: int64

In [65]:
class_distribution = (
    df_full.groupby('ClassId')['has_defect']
      .mean()
      .rename('defect_ratio')
)

print("Per-Class Defect Frequency", class_distribution, sep="\n")

Per-Class Defect Frequency
ClassId
1    0.134563
2    0.037054
3    0.772577
4    0.120162
Name: defect_ratio, dtype: float64


In [ ]:
print(f"No. of unique images:", df_full['ImageId'].nunique())
print(f"No. of images with no defects:", image_level_defects.value_counts().sum())

No. of unique images: 6666
No. of images with no defects: 6666


In [67]:
train_images_dir = os.path.join(DATA_DIR, "train_images")

missing_files = [
    img for img in df['ImageId'].unique()
    if not os.path.exists(os.path.join(train_images_dir, img))
]

print("Missing image files count:", len(missing_files))

Missing image files count: 0
